# CivBench Game Analysis

Analyze Civilization VI benchmark games played by AI agents.

**Setup**: Create a `.env` file in this directory (or the repo root `evals/.env`) with:
```
AZURE_SAS_TOKEN=<your-token>
```

In [ ]:
import sys
sys.path.insert(0, '../scripts')
import civbench_data as cb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Games Overview

In [ ]:
games = cb.list_games()
print(f'{len(games)} games ({games["admissible"].sum()} admissible)')
games[['model', 'scenario', 'turns', 'score', 'result', 'victory_type', 'winner', 'run_id']]

In [ ]:
# Win rates by model
admissible = games[games['admissible']]
summary = admissible.groupby('model').agg(
    games=('run_id', 'count'),
    wins=('result', lambda x: (x == 'victory').sum()),
    avg_turns=('turns', 'mean'),
    avg_score=('score', 'mean'),
).assign(win_rate=lambda d: (d['wins'] / d['games'] * 100).round(1))
summary

## 2. Single Game Deep-Dive

Pick a game from the table above and paste its `run_id` below.

In [ ]:
# Pick a game (default: first admissible)
if admissible.empty:
    print("No admissible games found — using all games instead.")
    admissible = games
RUN_ID = admissible.iloc[0]['run_id']
print(f'Analyzing: {RUN_ID}')

agent = cb.agent_turns(RUN_ID)
print(f'{len(agent)} turns')
agent[['turn', 'score', 'science', 'culture', 'gold_per_turn', 'military', 'cities']].tail(10)

In [ ]:
# Yield curves over time
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
metrics = [
    ('score', 'Score'),
    ('science', 'Science/turn'),
    ('culture', 'Culture/turn'),
    ('military', 'Military Strength'),
]
for ax, (col, label) in zip(axes.flat, metrics):
    if col in agent.columns:
        ax.plot(agent['turn'], agent[col], linewidth=1.5)
        ax.set_xlabel('Turn')
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.grid(alpha=0.3)
plt.suptitle(f'Agent Yields: {RUN_ID}', fontsize=13)
plt.tight_layout()
plt.show()

## 3. 8-Dimension Scoring

In [ ]:
scores = cb.score_game(RUN_ID)
score_df = pd.DataFrame([
    {'Dimension': dim, 'Score': r['score'], 'Details': r['details']}
    for dim, r in scores.items()
])
score_df

In [ ]:
# Radar chart
import numpy as np

dims = list(scores.keys())
vals = [scores[d]['score'] for d in dims]
angles = np.linspace(0, 2 * np.pi, len(dims), endpoint=False).tolist()
vals += vals[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.plot(angles, vals, 'o-', linewidth=2)
ax.fill(angles, vals, alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels([d.replace(' ', '\n') for d in dims], size=9)
ax.set_ylim(0, 100)
ax.set_title(f'Game Score: {RUN_ID}', pad=20)
plt.tight_layout()
plt.show()

## 4. Multi-Model Scorecard

Average scores across all admissible games per model. Takes a few minutes to compute (fetches all game data).

In [ ]:
summary_scores = cb.scorecard_summary()
summary_scores

In [ ]:
# Heatmap
if not summary_scores.empty:
    fig, ax = plt.subplots(figsize=(12, 4))
    data = summary_scores[cb.DIMENSIONS]
    im = ax.imshow(data.values, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
    ax.set_xticks(range(len(cb.DIMENSIONS)))
    ax.set_xticklabels([d.replace(' ', '\n') for d in cb.DIMENSIONS], rotation=0, ha='center', fontsize=9)
    ax.set_yticks(range(len(data.index)))
    ax.set_yticklabels(data.index)
    for i in range(len(data.index)):
        for j in range(len(cb.DIMENSIONS)):
            ax.text(j, i, f'{data.values[i, j]:.0f}', ha='center', va='center', fontsize=10)
    plt.colorbar(im, ax=ax, label='Score (0-100)')
    plt.title('Model Scorecard (mean across admissible games)')
    plt.tight_layout()
    plt.show()

## 5. Tool Usage Patterns

In [ ]:
log = cb.load_log(RUN_ID)
tool_calls = log[log['type'] == 'tool_call'] if 'type' in log.columns else log
print(f'{len(tool_calls)} tool calls over {agent["turn"].max()} turns')
if 'duration_s' in tool_calls.columns:
    print(f'Mean duration: {tool_calls["duration_s"].mean():.2f}s')
print()

# Top tools by frequency
tool_calls['tool'].value_counts().head(15)

In [ ]:
# Tool calls per turn
calls_per_turn = tool_calls.groupby('turn').size()
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(calls_per_turn.index, calls_per_turn.values, width=1, alpha=0.7)
ax.set_xlabel('Turn')
ax.set_ylabel('Tool Calls')
ax.set_title(f'Tool Calls per Turn: {RUN_ID}')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()